In [ ]:
# from tqdm import tqdm
import json
import os
from datetime import datetime

import torch
from torch_geometric.loader import DataLoader
# import torch.nn as nn

from extractor import PDBBindOrchestrator
from tokenizer import UniversalPDBBindDataset
# from model.model import UniversalHybridSlotModel
# from encoders.original_quantum_encoder import QuantumReUploadingLayer
# from encoders.trio_encoder import TrioEncoder
from evaluator import Evaluator
from splitter import PDBBindSplitter
# from loss_functions.loss_functions import get_loss_function
from utils import Utils
from model.model_builder import UHSMBuilder
from parsers.cnn_parser import CNNParser
from parsers.gnn_parser import GNNParser
from logger import *
from model.trainer import HybridTrainer

In [ ]:
with open('config.json', 'r') as f:
        config = json.load(f)

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
exp_name = f"{config['experiment_name']}_{timestamp}"

exp_run_dir = f"runs/{exp_name}"
exp_data_dir = f"datasets/{exp_name}" # Индивидуальная папка для датасетов!

os.makedirs(exp_run_dir, exist_ok=True)
os.makedirs(exp_data_dir, exist_ok=True)
os.makedirs("data/base_datasets", exist_ok=True) # Глобальный кэш

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Запуск на устройстве: {device}")

In [ ]:
trio_str = config['model']['graph_encoder']['available']['trio']['protein_ligand_pocket_encoders']
parsers = []
for i, char in enumerate(trio_str):
    is_lig = (i == 1)
    if char == 'C': parsers.append(CNNParser(is_ligand=is_lig))
    elif char == 'G': parsers.append(GNNParser(is_ligand=is_lig))
    elif char == 'N': parsers.append(None)
    else: logger.info("[CONFIGURATION] Unknown symbol {char} in the architecture description.")

In [ ]:
orchestrator = PDBBindOrchestrator(parsers, config)
orchestrator.extract_subset("refined")
df_refined = orchestrator.build_dataset(subset="refined")
df_core = orchestrator.build_dataset(subset="core")

In [ ]:
clean_refined = df_refined[~df_refined['pdb_id'].isin(df_core['pdb_id'])]

train_df, val_df = PDBBindSplitter.split(
    clean_refined,
    strategy=config["dataset"]["split_strategy"], 
    val_frac=config["dataset"]["val_frac"],
    seed=42
)

test_df = df_core

train_path = f"{exp_data_dir}/train.pickle"
val_path   = f"{exp_data_dir}/val.pickle"
test_path  = f"{exp_data_dir}/test_core.pickle"

train_df.to_pickle(train_path)
test_df.to_pickle(test_path)
val_df.to_pickle(val_path)

config["dataset"].update({
    "train_path": train_path,
    "val_path": val_path,
    "test_path": test_path,
})

In [ ]:
print(f"Эксперимент: {exp_name}")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test (Core): {len(df_core)}")

In [ ]:
train_ds = UniversalPDBBindDataset(config["dataset"]["train_path"], config)
test_ds = UniversalPDBBindDataset(config["dataset"]["test_path"], config)
val_ds   = UniversalPDBBindDataset(config["dataset"]["val_path"], config)

train_loader = DataLoader(train_ds, batch_size=config['dataset']['batch_size'], shuffle=True)
test_loader = DataLoader(test_ds, batch_size=config['dataset']['batch_size'], shuffle=False)
val_loader   = DataLoader(val_ds, batch_size=config['dataset']['batch_size'], shuffle=False)

In [ ]:
exp_dir = Utils.handle_metadata(config, train_ds, val_ds, test_ds)
model = UHSMBuilder.build_model_from_config(config)
evaluator = Evaluator(model, device)
print(f"Модель собрана. Общий выход энкодера: {model.graph_encoder.out_dim}")

In [ ]:
trainer = HybridTrainer(model, evaluator, config, device)
best_epoch, best_val_r = trainer.train(train_loader, val_loader, exp_dir)
trainer.test(test_loader, exp_dir, best_epoch)